Most of our datasets track each metric in overlapping five-year blocks (ie 2010-14, 2011-15, 2012-16, and so on). However, the tables for fine particles, median income, premature mortality, and crime rate do not do this, tracking their metrics by individual years instead. This notebook creates modified versions of each of those tables that do use the block system, using a window function that takes the average value from each group of five rows in the original table.

The median income table needs additional parsing, as it uses a different format from the others and also lacks any data for the year 2020, requiring manual interpolation.

The crime rate table also needs additional parsing, as the data is being taken from `neighborhood_indicators.csv` which contains an enormous amount of irrelevant data.

In [1]:
import pyspark.sql.functions as sf
from pyspark.sql import SparkSession
from pyspark.sql.window import Window

In [91]:
spark = SparkSession.builder.appName("blockify_years").getOrCreate()

particlesDF = spark.read.format("csv") \
              .option("header", True) \
              .option("infer_schema", True) \
              .load("../fine_particles.csv")

incomeDF = spark.read.format("csv") \
           .option("header", True) \
           .option("infer_schema", True) \
           .load("../median_income.csv")

mortalityDF = spark.read.format("csv") \
              .option("header", True) \
              .option("infer_schema", True) \
              .load("../mortality.csv")

indicDF = spark.read.format("csv") \
          .option("header", True) \
          .option("infer_schema", True) \
          .load("../neighborhood_indicators.csv")

In [92]:
# create fake data for the year 2020 by averaging the values for 2018, 2019, 2021, and 2022
fourYearWindow = Window.partitionBy("location_name").orderBy("timeframe").rowsBetween(-1,2)
rowsFor2020 = incomeDF.select("*") \
                      .filter("timeframe >= 2018 AND timeframe <= 2022") \
                      .withColumn("data", sf.avg("data").over(fourYearWindow)) \
                      .filter("timeframe = 2019") \
                      .withColumn("timeframe", sf.lit(2020))

# TODO: replace the CD names with district codes
    
# add the fake data, and rename the columns to match the format of the other tables
cleanedIncomeDF = incomeDF.union(rowsFor2020) \
                          .withColumnsRenamed({
                              "timeframe": "TimePeriod", 
                              "data": "median_income",
                              "location_name": "GeoID"
                          })

In [95]:
# filter out all the extraneous columns and region types
crimeDF = indicDF.filter("region_type = 'Community District'") \
                 .filter("crime_all_rt IS NOT NULL") \
                 .select("region_id", "year", "crime_all_rt") \
                 .withColumnsRenamed({
                     "year": "TimePeriod",
                     "region_id": "GeoID"
                 })

In [97]:
tableMapping = [
    (particlesDF, "Annual mean mcg/m3", "fine_particles_parsed"),
    (cleanedIncomeDF, "median_income", "median_income_parsed"),
    (mortalityDF, "Age-adjusted rate per 100,000", "mortality_parsed"),
    (crimeDF, "crime_all_rt", "crime_rate_parsed")
]

for df, colName, outName in tableMapping:
    lagWindow = Window.partitionBy("GeoID").orderBy("TimePeriod")
    blockAvgWindow = lagWindow.rowsBetween(-4, 0)

    blockified = df.select(
        "GeoID",
        sf.concat(
            sf.to_char(sf.lag("TimePeriod", 4).over(lagWindow), sf.lit("9999")),
            sf.lit("-"),
            sf.to_char(sf.try_subtract("TimePeriod", sf.lit(2000)), sf.lit("09"))
        ).alias("TimePeriod"),
        sf.round(sf.avg(colName).over(blockAvgWindow), 2).alias(colName)
    ).filter("TimePeriod IS NOT NULL")

    blockified.write.format("csv").mode("overwrite").save("../new_tables/"+outName, header=True)